In [1]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
from pathlib import Path

In [2]:
class OLSNormalEquation:

    def __init__(self):
        self.coef_ = None

    @staticmethod
    def has_perfect_multicollinearity(X):
        '''
        to check if design matrix X has perfect multicollinearity. perfect multicollinearity makes XTX singular, so (XTX)-1 does not exist
        '''

        rank = np.linalg.matrix_rank(X)
        n_cols = X.shape[1]
        return rank < n_cols

    def fit(self,X,y):
        '''
        input is design matrix X and response matrix y
        '''
        if self.has_perfect_multicollinearity(X):
            raise ValueError("Perfect multicollinearity detected")

        XtX = X.T @ X
        Xty = X.T @ y
        self.coef_ = np.linalg.inv(XtX) @ Xty
        return self

    def predict(self,X):
        if self.coef_ is None:
            raise ValueError("Model has not been fitted yet")

        return X @ self.coef_




start from the loss func which is the rss bowl
L = (y - y_hat)^2 = (y - b0 - b1x)^2
L = (y - X.T @ b)
L is a func of b -> L(b)

get gradient func del L
b0 = b0 - alpha * del L
b1 = b1 - alpha * del L
at each iteration calculate L
stop when L converges 

In [4]:
df = sm.datasets.longley.load_pandas().data


In [5]:
df.shape
df.head()

,TOTEMP,GNPDEFL,GNP,UNEMP,ARMED,POP,YEAR
0,60323.0,83.0,234289.0,2356.0,1590.0,107608.0,1947.0
1,61122.0,88.5,259426.0,2325.0,1456.0,108632.0,1948.0
2,60171.0,88.2,258054.0,3682.0,1616.0,109773.0,1949.0
3,61187.0,89.5,284599.0,3351.0,1650.0,110929.0,1950.0
4,63221.0,96.2,328975.0,2099.0,3099.0,112075.0,1951.0


In [6]:
y = df["TOTEMP"].to_numpy()
X = df.drop(columns="TOTEMP").to_numpy()
X = np.column_stack((
    np.ones(len(X)),
    X
))
print(X.shape)
print(y.shape)

(16, 7)
(16,)


In [7]:
X

array([[1.00000e+00, 8.30000e+01, 2.34289e+05, 2.35600e+03, 1.59000e+03,
        1.07608e+05, 1.94700e+03],
       [1.00000e+00, 8.85000e+01, 2.59426e+05, 2.32500e+03, 1.45600e+03,
        1.08632e+05, 1.94800e+03],
       [1.00000e+00, 8.82000e+01, 2.58054e+05, 3.68200e+03, 1.61600e+03,
        1.09773e+05, 1.94900e+03],
       [1.00000e+00, 8.95000e+01, 2.84599e+05, 3.35100e+03, 1.65000e+03,
        1.10929e+05, 1.95000e+03],
       [1.00000e+00, 9.62000e+01, 3.28975e+05, 2.09900e+03, 3.09900e+03,
        1.12075e+05, 1.95100e+03],
       [1.00000e+00, 9.81000e+01, 3.46999e+05, 1.93200e+03, 3.59400e+03,
        1.13270e+05, 1.95200e+03],
       [1.00000e+00, 9.90000e+01, 3.65385e+05, 1.87000e+03, 3.54700e+03,
        1.15094e+05, 1.95300e+03],
       [1.00000e+00, 1.00000e+02, 3.63112e+05, 3.57800e+03, 3.35000e+03,
        1.16219e+05, 1.95400e+03],
       [1.00000e+00, 1.01200e+02, 3.97469e+05, 2.90400e+03, 3.04800e+03,
        1.17388e+05, 1.95500e+03],
       [1.00000e+00, 1.04600

In [8]:
print("X min/max:", np.nanmin(X), np.nanmax(X))
print("y min/max:", np.nanmin(y), np.nanmax(y))

print("NaN in X:", np.isnan(X).any())
print("NaN in y:", np.isnan(y).any())

print("Inf in X:", np.isinf(X).any())
print("Inf in y:", np.isinf(y).any())

X min/max: 1.0 554894.0
y min/max: 60171.0 70551.0
NaN in X: False
NaN in y: False
Inf in X: False
Inf in y: False


In [9]:
norm = OLSNormalEquation()
norm.fit(X,y)
print(norm.coef_)
print(norm.predict(X))

[-3.48225864e+06  1.50618734e+01 -3.58191797e-02 -2.02022981e+00
 -1.03322687e+00 -5.11041047e-02  1.82915147e+03]
[60055.66497512 61216.01894971 60124.71783951 61597.11962768
 62911.29041526 63888.31622261 65153.05396435 63774.18536539
 66004.70023315 67401.61091316 68186.27393591 66552.06005207
 68810.55498044 69649.67631516 68989.0734921  70757.76282856]


In [14]:
model = sm.OLS(y, X)
results = model.fit()

print(results.summary())
y_hat = results.predict(X)
print(y_hat)

                            OLS Regression Results                            
Dep. Variable:                      y   R-squared:                       0.995
Model:                            OLS   Adj. R-squared:                  0.992
Method:                 Least Squares   F-statistic:                     330.3
Date:                Sat, 12 Sep 2026   Prob (F-statistic):           4.98e-10
Time:                        20:23:31   Log-Likelihood:                -109.62
No. Observations:                  16   AIC:                             233.2
Df Residuals:                       9   BIC:                             238.6
Df Model:                           6                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const      -3.482e+06    8.9e+05     -3.911      0.0